<a href="https://colab.research.google.com/github/imsahilahmed/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imsahilahmed/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*



**Lane: Refresh / Content Opportunity Scoring**

I'm choosing this lane because I've already spent Week 1 running the starter pipeline end
to end, and it directly answers this lane's question: *which pages should be reviewed first
for refresh, expansion, protection, pruning, or monitoring?* The pipeline already gave me a
transparent baseline (a hand-written scoring rule) and a trained model that clearly beats it
on Precision@50 — so I have early proof this lane is learnable, not just a hunch. I also found
a real secondary signal during EDA (content_type affects CTR independent of position), which
suggests there's more headroom in this lane to explore over the next 7 weeks — e.g. adding
content_type as a baseline reason code or model feature.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*


**Research question:** Which content pages should an editor review first this week, given
limited review capacity?

**Decision improved:** Instead of an editor guessing which pages to check (or working through
them in a random or alphabetical order), they get a ranked queue ordered by evidence of risk
or opportunity.

**Who acts, and what do they do:** A content editor or SEO strategist with limited weekly
capacity (e.g. can only review ~20-50 pages) pulls from the top of the ranked queue and takes
one of a few actions per page: refresh the content, expand it, protect it as-is, prune it, or
just monitor it.

**Cost of a wrong call:**
- If a page is flagged high-priority but isn't actually a problem, the editor wastes limited
  review time on it — time that could've gone to a page that really needed attention.
- If a genuinely declining page is ranked low (missed), it keeps losing traffic quietly until
  the next review cycle catches it — which could be weeks away.

**Why data/ML helps, not just a rule:** A single signal doesn't cleanly separate "needs review"
from "fine." I already found that word count barely differs between declining and growing pages,
and search volume barely predicts real traffic — so simple one-variable rules miss a lot. The
starter baseline (a hand-written rule combining several signals) only gets ~12 of its top 50
picks right; a random forest trained on the same signals gets ~37 right. That's evidence the
pattern is real but too tangled across multiple signals for a human to hand-write well — which
is exactly when ML earns its place.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [5]:
!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/flyrank-ml-internship-starter/outputs/refresh_que

In [6]:
import json, pandas as pd

# Number 1 & 2: baseline vs model, from your Week 1 pipeline run
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"Baseline rule Precision@50: {base:.3f}  (~{round(base*50)} of top 50 correct)")
print(f"Random forest Precision@50: {rf:.3f}  (~{round(rf*50)} of top 50 correct)")
print(f"Lift: {rf/base:.1f}x over the hand-written rule\n")

# Number 3: a real EDA signal showing there's more headroom in this lane
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
visible = df[df["impressions_90d"] >= 100]
ctr_by_type = visible.groupby("content_type")["ctr"].mean().sort_values()
print("CTR range across content types (impressions >= 100):")
print(ctr_by_type.round(3).to_string())

Baseline rule Precision@50: 0.240  (~12 of top 50 correct)
Random forest Precision@50: 0.740  (~37 of top 50 correct)
Lift: 3.1x over the hand-written rule

CTR range across content types (impressions >= 100):
content_type
comparison article    0.134
keyword article       0.252
feedly article        0.706


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*



**What I can claim:**
- These results are *observed* and *directional* — patterns seen in a 30,000-row anonymized
  starter slice, validated with a client-holdout split.
- The ranking is *decision-support*: it helps a human prioritize limited review time, it does
  not make the decision for them.
- Claims like "the model beats the baseline on this data" are measured and reproducible from
  the notebook.

**What I cannot claim:**
- I cannot claim a refresh will *cause* a page to recover — that would require a real
  experiment (e.g. an A/B test), not this observational data.
- I cannot claim to have "found" or "predicted" anything about Google's ranking algorithm —
  I'm working with observable signals (impressions, clicks, position), not the algorithm itself.
- I will not use `trend_direction` or `trend_pct` as model features, since the label
  (`is_declining_label`) is derived directly from them — using them as features would be
  circular, not a real discovery.
- I cannot generalize these numbers to the full ~79M-row warehouse yet; this is a 30k-row
  starter slice, and the result needs to be re-earned on the full data with proper validation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.